# Deploy an XGBoost model on TrueFoundry with Triton, no inference code

- Flow: **train → log model → package → build & push image → deploy → infer**
- Inference code: autogenerated by TFY CLI
- Image: build and pushed to registry first, never built by TrueFoundry
- Docs: https://www.truefoundry.com/docs/xgboost
  (schema, serialization formats, and the `Incomplete model manifest` troubleshooting)

> **Needs a recent control plane.** Until
> [mlfoundry-server#1196](https://github.com/truefoundry/mlfoundry-server/pull/1196) (merged
> 2026-09-28) step 3 returned 400 `Model framework is not supported for deployment` for every
> XGBoost model: the server compared the manifest against the wrong `XGBoostFramework` class,
> so the `isinstance` check never matched. Sklearn was unaffected. On an older control plane
> this notebook still fails there; log as `SklearnFramework` as a workaround.

---

## 0. Setup

> **Kernel: `Python 3.12 (TFY Triton)`** — venv at `~/work/.venvs/tfy-triton-py312`.
> Not a preference: logging from 3.14 makes step 3 fail with
> `Python version 3.14 is not supported for Triton deployment`. The generated Dockerfile takes
> its base image from the *logged model version's* Python version, and the image installs
> `nvidia-pytriton`, which ships no wheels past 3.12 and needs numpy < 2.
> Documented at <https://www.truefoundry.com/docs/xgboost>: *"Python version < 3.8 and > 3.12
> is not supported for Triton deployment"*, and numpy must be < 2.0.0.

In [ ]:
# %pip install -U truefoundry xgboost joblib requests

In [ ]:
import os
import sys
import time
from pathlib import Path

import joblib
import requests
from sklearn.datasets import load_iris
from xgboost import XGBClassifier

from truefoundry.ml import get_client, XGBoostFramework, xgboost_infer_schema
from truefoundry.common.session import Session
from truefoundry_sdk import TrueFoundry

# `!` inherits the notebook process PATH, which lacks the kernel venv's bin/.
os.environ["PATH"] = f"{Path(sys.executable).parent}{os.pathsep}{os.environ['PATH']}"

In [ ]:
TFY_HOST = "https://platform-admin.live-demo.truefoundry.cloud"
ML_REPO = "cx-team"
CLUSTER = "tfy-azure"
WORKSPACE = "seldon-tf"
DOMAIN = "az-apps.live-demo.truefoundry.cloud"  # per-cluster: tfy-azure
WS_FQN = f"{CLUSTER}:{WORKSPACE}"

DOCKER_USER = "mkozubal"
TAG = "1"
PORT = 8000
READY_TIMEOUT = 300

NAME = "iris-xgb-triton"
MODEL_FILE = f"{NAME}.joblib"
SVC = f"{NAME}-svc"
IMAGE = f"{DOCKER_USER}/{SVC}:{TAG}"
HOST = f"{SVC}-{WORKSPACE}-{PORT}.{DOMAIN}"

## 1. Train and save

In [ ]:
X, y = load_iris(return_X_y=True)
model = XGBClassifier(n_estimators=10).fit(X, y)
joblib.dump(model, MODEL_FILE)

## 2. Log model 
- schema goes here
- `model_filepath` + `serialization_format` + `model_schema` are what make the version deployable

In [ ]:
client = get_client()

# XGBoost supports only `predict`, so no infer_method_name
framework = XGBoostFramework(
    model_filepath=MODEL_FILE,
    serialization_format="joblib",
    model_schema=xgboost_infer_schema(model_input=X, model=model),
)

FQN = client.log_model(
    ml_repo=ML_REPO, name=NAME, model_file_or_folder=MODEL_FILE, framework=framework,
).fqn
print(FQN)

## 3. Generate the serving package

`tfy deploy-init` writes `server.py`, `infer.py`, `requirements.txt`, `deploy.py`, `README.md` locally.
Nothing is built and nothing is deployed — same command the UI Deploy button hands you.

In [ ]:
!tfy deploy-init model --name {SVC} --model-version-fqn '{FQN}' --workspace-fqn '{WS_FQN}' --model-server triton
assert _exit_code == 0

In [ ]:
!tree {SVC}

## 4. Build and push the image

Unlike FastAPI, the Triton codegen **does** emit a `Dockerfile` (`python:3.12` + `nvidia-pytriton`),
so there is nothing to write — build it as-is and push to Docker Hub. `deploy.py` goes unused.

In [ ]:
!docker build --platform linux/amd64 -t {IMAGE} {SVC}
assert _exit_code == 0

!docker push {IMAGE}
assert _exit_code == 0

## 5. Deploy as a service

- image: prebuilt (`"image": {"type": "image", ...}`)
- `artifacts_download`: pulls the model version into `$MODEL_PATH` (the image is model-free)
- auth reuses the `tfy login` session (no PAT in the notebook)
- `create_or_update`: returns a deployment, so delete later with `app.data.application_id`

In [ ]:
tfy = TrueFoundry(base_url=TFY_HOST, api_key=Session.new().access_token)

app = tfy.applications.create_or_update(manifest={
    "name": SVC, "type": "service", "workspace_fqn": WS_FQN,
    "image": {"type": "image", "image_uri": IMAGE},
    "artifacts_download": {"artifacts": [{"type": "truefoundry-artifact",
        "artifact_version_fqn": FQN, "download_path_env_variable": "MODEL_PATH"}]},
    "ports": [{"port": PORT, "protocol": "TCP", "expose": True,
               "app_protocol": "http", "host": HOST}],
    "resources": {"cpu_request": 0.5, "cpu_limit": 0.5,
                  "memory_request": 1000, "memory_limit": 1000,
                  "ephemeral_storage_request": 2000, "ephemeral_storage_limit": 2000},
    "replicas": 1,
})
print(SVC, app.data.application_id)

## 6. Call the model

- KServe v2: `POST /v2/models/<model_name>/infer`, `<model_name>` = the name `server.py` binds (no Swagger)
- Poll `/v2/models/<model_name>/ready`, **not** `/v2/health/ready` — server-ready lands ~10s before pytriton binds the model

In [ ]:
deadline = time.time() + READY_TIMEOUT
while time.time() < deadline:
    try:
        if requests.get(f"https://{HOST}/v2/models/{SVC}/ready", timeout=5).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    raise TimeoutError(f"{SVC} not ready after {READY_TIMEOUT}s")

print(SVC, "ready")

In [ ]:
r = requests.post(
    f"https://{HOST}/v2/models/{SVC}/infer",
    json={"inputs": [{"name": "input_0", "shape": [1, 4], "datatype": "FP32",
                      "data": [[5.1, 3.5, 1.4, 0.2]]}]},
    timeout=30,
)
print(r.status_code, r.text)

The same call as a copy-pasteable `curl`, for testing outside the notebook:

In [ ]:
print(f'''curl -X POST https://{HOST}/v2/models/{SVC}/infer \\
  -H 'Content-Type: application/json' \\
  -d '{{"inputs": [{{"name": "input_0", "shape": [1, 4], "datatype": "FP32", "data": [[5.1, 3.5, 1.4, 0.2]]}}]}}' ''')

## Cleanup
- service, local package, model file
- model version stays — a re-run logs a new one
- image left alone — `docker build`/`push` overwrite tag `1` in place

In [ ]:
# service
APP_ID = app.data.application_id
try:
    tfy.applications.delete(id=APP_ID)
except Exception:
    tfy.applications.cancel_deployment(
        id=APP_ID, deployment_id=tfy.applications.get(id=APP_ID).data.last_deployment_id)
    tfy.applications.delete(id=APP_ID)
print("deleted", SVC)

# local package, model file
!rm -rf {SVC} {MODEL_FILE}
